In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import glob
import pickle
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import seaborn as sns

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection

In [2]:
# load saved df for model fitting results
df_model_fitting_relevant = pd.read_pickle(
    os.path.expanduser("~/capsule/results/df_model_fitting_relevant_250714.pkl")
)

# simulate sessions with fitted parameters

In [ ]:
# per subject_id, bootstrap sampling a fitted session
# use only current_stage_actual in ['STAGE_FINAL', 'GRADUATED']

def get_bootstrap_sample(df, n_samples=1):
    """Get a bootstrap sample from the fitted sessions."""
    df_filtered = df[df['current_stage_actual'].isin(['STAGE_FINAL', 'GRADUATED'])]
    if df_filtered.empty:
        raise ValueError("No valid sessions found for bootstrapping.")
    return df_filtered.sample(n=n_samples, replace=True)

def get_bootstrap_samples(df, n_samples=100):
    """Get bootstrap samples for each subject_id."""
    bootstrap_samples = {}
    for subject_id in df['subject_id'].unique():
        df_subject = df[df['subject_id'] == subject_id]
        if not df_subject.empty:
            bootstrap_samples[subject_id] = get_bootstrap_sample(df_subject, n_samples)
    return bootstrap_samples

# use the sampled session fitted parameters to simulate a forager in the same type of task 
def simulate_forager(task_type, bootstrap_samples, n_trials=100):
    """Simulate a forager using the bootstrap samples."""
    foragers = {}
    for subject_id, sample in bootstrap_samples.items():
        if task_type == 'coupled':
            task = CoupledBlockTask()
        elif task_type == 'uncoupled':
            task = UncoupledBlockTask()
        else:
            raise ValueError("Invalid task type. Choose 'coupled' or 'uncoupled'.")
        
        forager = ForagerCollection.from_fitted_params(
            sample['fitted_params'].iloc[0], task=task
        )
        foragers[subject_id] = forager.simulate(n_trials=n_trials)
    return foragers